In [0]:
from pyspark.sql import functions as F
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = spark.sql("SELECT current_schema()").first()[0]
expected_tables = {
   "raw_data": f"{catalog_name}.{schema_name}.fraud_transactions_raw",
   "training_data": f"{catalog_name}.{schema_name}.fraud_transaction_train",
   "test_data": f"{catalog_name}.{schema_name}.fraud_transaction_test",
   "predictions": f"{catalog_name}.{schema_name}.fraud_predictions_test",
   "normal_drift": f"{catalog_name}.{schema_name}.fraud_drift_normal",
   "simulated_drift": f"{catalog_name}.{schema_name}.fraud_drift_simulated",
   "performance": f"{catalog_name}.{schema_name}.fraud_model_performance",
   "model_registry": f"{catalog_name}.{schema_name}.fraud_model_run_registry"
}
print("Catalog:", catalog_name)
print("Schema:", schema_name)

Catalog: workspace
Schema: default


In [0]:
import pandas as pd

validation_results = []
for name, table_name in expected_tables.items():
   try:
       row_count = spark.table(table_name).count()
       validation_results.append({
           "check": name,
           "table_name": table_name,
           "row_count": row_count,
           "status": "PASS" if row_count > 0 else "FAIL"
       })
   except Exception as exc:
       validation_results.append({
           "check": name,
           "table_name": table_name,
           "row_count": 0,
           "status": "FAIL"
       })
       print(f"Could not validate {table_name}: {exc}")
validation_pdf = pd.DataFrame(validation_results)
display(validation_pdf)
failed_checks = validation_pdf[
   validation_pdf["status"] != "PASS"
]
if not failed_checks.empty:
   raise ValueError(
       f"Pipeline validation failed for: "
       f"{failed_checks['check'].tolist()}"
   )
print("All pipeline table checks passed.")

check,table_name,row_count,status
raw_data,workspace.default.fraud_transactions_raw,284807,PASS
training_data,workspace.default.fraud_transaction_train,227845,PASS
test_data,workspace.default.fraud_transaction_test,56962,PASS
predictions,workspace.default.fraud_predictions_test,56962,PASS
normal_drift,workspace.default.fraud_drift_normal,30,PASS
simulated_drift,workspace.default.fraud_drift_simulated,30,PASS
performance,workspace.default.fraud_model_performance,5,PASS
model_registry,workspace.default.fraud_model_run_registry,1,PASS


All pipeline table checks passed.


In [0]:
registry_df = spark.table(
   expected_tables["model_registry"]
)
display(registry_df)
assert registry_df.count() > 0, "No model run found"
latest_run_id = (
   registry_df
   .orderBy(F.desc("registered_at_utc"))
   .first()["run_id"]
)
assert latest_run_id, "The model run ID is empty"
print("Latest model run:", latest_run_id)
print("Model registry validation passed.")

run_id,model_name,registered_at_utc
b3ca314766bc4afbb27d9bb58180dfbf,fraud_logistic_regression,2026-10-11T10:04:58.772978+00:00


Latest model run: b3ca314766bc4afbb27d9bb58180dfbf
Model registry validation passed.
